# Teaching Computers to See : classification Cats vs Dogs avec un CNN

Objectif : construire, entraîner et évaluer un Convolutional Neural Network capable de distinguer des images de chats et de chiens, puis essayer d'améliorer ses performances.

## Step 1 : Introduction aux CNN

Un Convolutional Neural Network est un réseau de neurones conçu pour les images. Un réseau dense classique traite chaque pixel indépendamment et demande énormément de paramètres. Un CNN exploite au contraire la structure spatiale de l'image : des pixels voisins sont liés entre eux.

- **Convolutional layer** : de petits filtres (kernels, par exemple 3×3) glissent sur l'image et calculent un produit scalaire à chaque position. Chaque filtre apprend à détecter un motif (bord, texture, forme) et produit une feature map. Les mêmes poids sont partagés sur toute l'image, d'où beaucoup moins de paramètres qu'une couche dense.
- **Pooling layer** : le MaxPooling garde la valeur maximale de chaque zone (par exemple 2×2). Il réduit la taille spatiale, donc le coût de calcul, et rend le modèle moins sensible aux petits décalages.
- **Activation function** : la ReLU (max(0, x)) introduit la non-linéarité, sans laquelle le réseau ne serait qu'une combinaison linéaire. Son gradient ne s'annule pas pour les valeurs positives, ce qui limite le vanishing gradient.
- **Fully connected layer** : après aplatissement (Flatten), des couches denses combinent les caractéristiques extraites pour produire la prédiction finale.

Les premières couches apprennent des motifs simples (bords, couleurs), les couches profondes des concepts plus abstraits (yeux, oreilles, museau).

# Import

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models

## Début du code, récupération des images et taille
## Step 2 : Exploration du dataset

On définit les chemins vers les dossiers `train` et `validation`, puis on compte le nombre d'images de chaque classe (cats, dogs) dans chacun. On affiche aussi la taille de 20 images de chats pour vérifier si elles ont toutes les mêmes dimensions.

In [ ]:
base_dir = "data/cats_and_dogs_filtered"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "validation")

for split, path in [("train", train_dir), ("validation", val_dir)]:
    for cls in ["cats", "dogs"]:
        folder = os.path.join(path, cls)
        print(f"{split}/{cls}: {len(os.listdir(folder))} images")

sample_dir = os.path.join(train_dir, "cats")
sizes = [Image.open(os.path.join(sample_dir, f)).size for f in os.listdir(sample_dir)[:20]]
print(sizes)

### Analyse

Le dossier `train` contient 2000 images (1000 cats, 1000 dogs) et le dossier `validation` 1000 images (500 cats, 500 dogs). Les classes sont parfaitement équilibrées : l'accuracy sera donc une métrique fiable, et aucune pondération des classes n'est nécessaire.

Les images sont en couleur (3 canaux RGB), mais leurs dimensions varient d'une image à l'autre. Un CNN demande une entrée de taille fixe, donc toutes les images seront redimensionnées en 150×150 au chargement.

### Visualisation de quelques images

On affiche 4 images tirées au hasard pour chaque classe (cats sur la première ligne, dogs sur la seconde), avec leurs dimensions en titre, pour se faire une idée concrète des données.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, cls in enumerate(["cats", "dogs"]):
    folder = os.path.join(train_dir, cls)
    for ax, fname in zip(axes[row], random.sample(os.listdir(folder), 4)):
        img = Image.open(os.path.join(folder, fname))
        ax.imshow(img)
        ax.set_title(f"{cls} {img.size}")
        ax.axis("off")
plt.tight_layout()
plt.show()

On observe une grande variété dans les images : cadrages différents, arrière-plans plus ou moins encombrés, animaux partiellement visibles ou accompagnés d'humains, éclairages variables. Les dimensions changent aussi d'une image à l'autre, ce qui confirme la nécessité du redimensionnement. La tâche n'est donc pas triviale : le modèle devra apprendre des caractéristiques propres à l'animal, et non à l'arrière-plan.

### Split 80/20 et chargement des données

On charge les images avec `image_dataset_from_directory`, qui associe automatiquement chaque image à sa classe d'après le nom de son dossier.

- **Redimensionnement en 150×150** : assez grand pour garder les détails utiles (oreilles, museau, pelage), assez petit pour que l'entraînement reste raisonnable.
- **Split 80/20 du dossier `train`** : 80 % pour l'entraînement (1600 images) et 20 % pour le test (400 images). Le test set n'est utilisé qu'à l'évaluation finale, ce qui donne une mesure honnête des

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 100

train_ds, test_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="both",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

print(train_ds.class_names)
for images, labels in train_ds.take(1):
    print(images.shape, labels.shape)
    counts = {}

for split, path in [("train", train_dir), ("validation", val_dir)]:
    for cls in ["cats", "dogs"]:
        counts[f"{split}/{cls}"] = len(os.listdir(os.path.join(path, cls)))

plt.figure(figsize=(8, 4))
plt.bar(counts.keys(), counts.values(), color=["tab:blue", "tab:orange", "tab:blue", "tab:orange"])
plt.title("Distribution des classes")
plt.ylabel("Nombre d'images")
plt.show()

Keras affiche "Using 400 files for validation" pour le deuxième sous-ensemble du split : dans notre cas, ces 400 images constituent bien le **test set**, la validation étant le dossier séparé de 1000 images.

Les classes sont détectées dans l'ordre alphabétique : `cats` = 0 et `dogs` = 1. Un batch a la forme `(100, 150, 150, 3)` : 100 images de 150×150 pixels sur 3 canaux RGB. Les labels ont la forme `(100, 1)`, une valeur binaire par image. L'input shape du modèle sera donc `(150, 150, 3)`.

## Step 3 : Construction du CNN

Pour notre modèle, on a suivi la structure demandée (4 blocs de convolution, du pooling, de la ReLU et une couche dense en sortie), mais en essayant de faire des choix adaptés à notre situation : un petit dataset de 1600 images d'entraînement.

On commence par une couche `Rescaling` qui ramène les pixels entre 0 et 1. Les réseaux apprennent mieux avec des valeurs petites, et comme cette étape fait partie du modèle, elle sera aussi appliquée automatiquement quand on fera des prédictions.

Chaque bloc contient une convolution 3×3 suivie d'un MaxPooling 2×2. On a choisi de doubler le nombre de filtres à chaque bloc (16, 32, 64, puis 128) : au début, le réseau cherche des choses simples comme des bords ou des couleurs, et il n'a pas besoin de beaucoup de filtres pour ça. Plus on avance, plus les motifs deviennent complexes (une oreille, un museau), et plus il faut de filtres pour les représenter. Commencer à 16 plutôt qu'à 32 permet aussi de garder un modèle léger.

On utilise `padding="same"` pour que les convolutions ne rognent pas les bords de l'image. Sans ça, chaque convolution enlève un pixel de chaque côté, et un chat placé au bord de la photo pourrait perdre des informations. Ici, c'est uniquement le pooling qui réduit la taille, ce qui rend l'évolution des dimensions plus simple à suivre.

La ReLU est utilisée partout comme activation : elle est très simple à calculer et évite le problème du vanishing gradient qu'on aurait avec une sigmoid dans les couches cachées.

Pour la partie fully connected, on a mis une seule couche Dense de 128 neurones. On aurait pu en mettre 512, mais vu le peu d'images qu'on a, un modèle trop gros risque surtout d'apprendre les images par cœur. Enfin, la sortie est un seul neurone avec une sigmoid : il renvoie une probabilité entre 0 et 1, et on considère que l'image est un chien si elle dépasse 0.5.

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    layers.Rescaling(1.0 / 255),

    layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

### Analyse du modèle

Le résumé se lit de haut en bas, dans l'ordre où l'image traverse le réseau. Dans la colonne Output Shape, le `None` correspond à la taille du batch, qui n'est pas fixée à l'avance : les trois autres valeurs sont la hauteur, la largeur et le nombre de canaux.

| Couche | Sortie | Paramètres | Rôle |
|---|---|---|---|
| Rescaling | 150 × 150 × 3 | 0 | Ramène les pixels entre 0 et 1. Simple calcul fixe, rien à apprendre. |
| Conv2D (16) | 150 × 150 × 16 | 448 | 16 filtres détectent des motifs simples (bords, couleurs). La taille est conservée grâce au `padding="same"`. |
| MaxPooling | 75 × 75 × 16 | 0 | Divise la taille par 2 en gardant les activations les plus fortes. |
| Conv2D (32) | 75 × 75 × 32 | 4 640 | 32 filtres combinent les motifs précédents en textures et formes simples. |
| MaxPooling | 37 × 37 × 32 | 0 | 75 / 2 arrondi à l'inférieur. |
| Conv2D (64) | 37 × 37 × 64 | 18 496 | Des motifs plus complexes, comme des parties d'animal. |
| MaxPooling | 18 × 18 × 64 | 0 | Nouvelle division par 2. |
| Conv2D (128) | 18 × 18 × 128 | 73 856 | Les motifs les plus abstraits (oreilles, museau, forme de la tête). |
| MaxPooling | 9 × 9 × 128 | 0 | Il reste 128 feature maps de 9 × 9. |
| Flatten | 10 368 | 0 | Déplie le bloc 9 × 9 × 128 en un vecteur, pour pouvoir le passer à une couche dense. |
| Dense (128) | 128 | 1 327 232 | Combine toutes les caractéristiques extraites pour prendre la décision. C'est la couche la plus lourde. |
| Dense (1) | 1 | 129 | Neurone de sortie avec sigmoid : la probabilité que l'image soit un chien. |

Grâce au `padding="same"`, les convolutions gardent la même taille d'image et seul le pooling la divise par 2 (de 150 × 150 à 75, puis 37, 18 et enfin 9 × 9). En parallèle, le nombre de canaux augmente de 3 à 128 : le réseau perd en résolution spatiale mais gagne en richesse d'information, chaque position décrivant des caractéristiques de plus en plus abstraites.

Le modèle compte 1 424 801 paramètres au total, tous entraînables, pour environ 5,4 Mo.

La répartition des paramètres est très déséquilibrée. Les quatre couches de convolution en totalisent moins de 100 000, alors que la couche Dense après le Flatten en contient à elle seule 1 327 232, soit environ 93 %. Les convolutions sont économes parce qu'elles réutilisent les mêmes petits filtres sur toute l'image, alors que la couche dense connecte chacune des 10 368 valeurs à chacun des 128 neurones. C'est donc surtout cette couche qui pourrait mémoriser les images d'entraînement au lieu d'apprendre des caractéristiques générales. On s'attend à voir un peu d'overfitting à l'entraînement.

## Step 4 : Entraînement du modèle

On entraîne le modèle :
- sur 15 epochs, la limite haute demandée, pour bien voir l'évolution des courbes et repérer un éventuel overfitting, qui n'apparaît souvent qu'après plusieurs epochs
- avec un batch size de 100, imposé par le sujet

À chaque epoch, le modèle est évalué sur le dossier de validation, pour comparer ce qu'il apprend sur les images d'entraînement avec ce qu'il réussit sur des images qu'il ne voit pas pendant l'apprentissage.

Avant de lancer l'entraînement, on ajoute `cache()` et `prefetch()` sur les datasets : les images sont gardées en mémoire après la première epoch, et le batch suivant est préparé pendant que le modèle travaille sur le batch actuel. Comme on est sur CPU, ça fait gagner pas mal de temps.

In [ ]:
EPOCHS = 15

train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
)

### Courbes d'apprentissage

On trace l'accuracy et la loss au fil des epochs, pour l'entraînement et la validation. On les met dans une fonction pour pouvoir la réutiliser à la Step 6 et comparer les versions du modèle.

In [ ]:
def plot_history(history, title):
    hist = history.history
    epochs_range = range(1, len(hist["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, hist["accuracy"], label="train")
    axes[0].plot(epochs_range, hist["val_accuracy"], label="validation")
    axes[0].set_title(f"{title} : accuracy")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(epochs_range, hist["loss"], label="train")
    axes[1].plot(epochs_range, hist["val_loss"], label="validation")
    axes[1].set_title(f"{title} : loss")
    axes[1].set_xlabel("epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_history(history, "Baseline")

À la fin des 15 epochs, le modèle atteint 88,3 % d'accuracy sur l'entraînement, mais seulement 73,7 % sur la validation. Cet écart de 14,6 points montre qu'il réussit beaucoup mieux sur les images qu'il connaît que sur des images nouvelles : il fait de l'**overfitting**. Ce n'est pas de l'underfitting, puisqu'il obtient de bons résultats sur les données d'entraînement.

Les courbes montrent à quel moment ça commence. Pendant les 4 premières epochs, l'entraînement et la validation progressent ensemble, d'environ 50 % (quasiment le hasard) à 67 %. La validation est même légèrement au-dessus de l'entraînement aux epochs 2 et 3. C'est normal : l'accuracy d'entraînement est une moyenne calculée pendant l'epoch, alors que le modèle s'améliore encore, tandis que la validation est mesurée à la fin de l'epoch, avec le modèle déjà mis à jour.

À partir de l'epoch 5, l'entraînement passe devant et l'écart ne fait plus que grandir. L'accuracy d'entraînement monte régulièrement jusqu'à 88 %, alors que celle de validation oscille entre 70 et 74 % sans vraiment progresser. C'est encore plus visible sur la loss : les deux courbes se croisent vers l'epoch 6, puis celle d'entraînement continue de baisser jusqu'à 0,30, tandis que celle de validation reste bloquée entre 0,53 et 0,57, avec son minimum à l'epoch 11 (0,535). Les zigzags de la validation montrent que le modèle ne s'améliore plus vraiment : il varie légèrement d'une epoch à l'autre, sans tendance claire.

Ça s'explique par ce qu'on avait vu à la Step 3 : seulement 1600 images d'entraînement pour plus de 1,4 million de paramètres, dont 93 % dans une seule couche dense, et aucune régularisation. Le modèle voit en plus exactement les mêmes images à chaque epoch, ce qui lui permet de les mémoriser petit à petit au lieu d'apprendre des caractéristiques générales.

L'overfitting reste modéré, sans doute grâce à nos choix pour garder le modèle léger (Dense de 128, learning rate réduit). Il limite quand même les performances : à partir de l'epoch 6 environ, continuer l'entraînement n'apporte presque plus rien en validation. C'est ce que la data augmentation et le dropout devraient améliorer à la Step 6.

## Step 5 : Évaluation sur le test set

Jusqu'ici, on a seulement regardé la validation. On évalue maintenant le modèle sur les 400 images du test set, qu'il n'a jamais vues, ni pendant l'entraînement ni pour ajuster quoi que ce soit. C'est la mesure la plus honnête de ses performances.

Pour chaque image, le modèle renvoie une probabilité entre 0 et 1 : au-dessus de 0.5, on considère que c'est un chien, sinon un chat. On récupère les prédictions et les vrais labels dans la même boucle, pour être sûrs qu'ils restent bien alignés.

On regarde ensuite :
- la **matrice de confusion**, pour voir où le modèle se trompe
- l'**accuracy** : la proportion d'images bien classées
- la **precision** : parmi les images que le modèle classe "dog", combien sont vraiment des chiens
- le **recall** : parmi tous les vrais chiens, combien le modèle en retrouve
- le **F1-score** : un compromis entre precision et recall

In [ ]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


def evaluate_model(model, dataset, title):
    y_true, y_prob = [], []
    for images, labels in dataset:
        y_true.append(labels.numpy().ravel())
        y_prob.append(model.predict(images, verbose=0).ravel())
    y_true = np.concatenate(y_true).astype(int)
    y_pred = (np.concatenate(y_prob) >= 0.5).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["cats", "dogs"]).plot(cmap="Blues")
    plt.title(f"{title} : confusion matrix")
    plt.show()

    results = {
        "model": title,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }
    for key in ["accuracy", "precision", "recall", "f1"]:
        print(f"{key}: {results[key]:.4f}")
    return results


baseline_results = evaluate_model(model, test_ds, "Baseline")

### Analyse des résultats

Sur les 400 images du test set, le modèle obtient une accuracy de **77,8 %**, un peu au-dessus de ce qu'on avait en validation (73,7 %). Ce petit écart n'est pas surprenant : le test set ne contient que 400 images, donc quelques images bien ou mal classées suffisent à faire bouger le score de plusieurs points. Les deux résultats restent du même ordre et confirment que le modèle généralise moins bien qu'il n'apprend (88 % sur l'entraînement).

On remarque aussi que le test set n'est pas parfaitement équilibré (182 chats, 218 chiens), parce que le split 80/20 est aléatoire. L'écart reste faible, donc l'accuracy reste une métrique fiable.

La matrice de confusion est plus intéressante que l'accuracy seule. Le modèle reconnaît très bien les chats (157 sur 182, soit 86 %), mais il a beaucoup plus de mal avec les chiens : 64 chiens sur 218 sont classés comme des chats, alors qu'à l'inverse seulement 25 chats sont pris pour des chiens. Le modèle a donc tendance à répondre "chat" quand il hésite.

Les métriques le confirment :
- **Precision de 86 %** : quand le modèle dit "chien", il a raison dans 86 % des cas. Il ne se trompe pas souvent dans ce sens.
- **Recall de 70,6 %** : en revanche, il ne retrouve que 70,6 % des vrais chiens. Près d'un chien sur trois lui échappe.
- **F1-score de 77,6 %** : il résume ce déséquilibre entre une bonne precision et un recall plus faible.

Une explication possible est que les photos de chiens sont plus variées que celles de chats : les races de chiens ont des tailles, des formes de tête et des pelages très différents, alors que les chats se ressemblent davantage. Avec seulement 1000 chiens à l'entraînement, le modèle n'a pas vu assez d'exemples pour couvrir toute cette diversité, et tout ce qui ne ressemble pas assez à un chien "typique" finit classé comme chat.

Au final, le modèle fait nettement mieux que le hasard, mais il reste limité par l'overfitting vu à la Step 4 et par ce biais vers la classe "cat". La data augmentation devrait aider sur les deux points, en montrant au modèle des versions plus variées des images à chaque epoch.